## Merge archaic results and save benchmark entry

In [3]:
# ── One-off repair: merge existing split parquet files ────────────────────────
from pathlib import Path
import pandas as pd
import json
from datetime import date

from src.data import REPO_ROOT

for run_dir in (REPO_ROOT / 'models' / 'evaluation').iterdir():
    if not run_dir.is_dir():
        continue
    rag_f  = run_dir / 'rag_results.parquet'
    base_f = run_dir / 'base_results.parquet'
    if not (rag_f.exists() and base_f.exists()):
        continue

    print(f'> Merging {run_dir.name}...')
    rag_df  = pd.read_parquet(rag_f)
    base_df = pd.read_parquet(base_f)

    results = rag_df.merge(
        base_df[['question', 'is_correct', 'pred', 'latency_s']],
        on='question', suffixes=('_rag', '_base'),
    )
    results.to_parquet(run_dir / 'results.parquet', index=False)

    rag_acc  = results['is_correct_rag'].mean() * 100
    base_acc = results['is_correct_base'].mean() * 100
    scores   = rag_df['top_score'].dropna()

    config = {
        'embedding':      run_dir.name.split('__')[0],
        'llm':            run_dir.name.split('__')[1],
        'n_eval':         len(results),
        'rag_accuracy':   round(rag_acc, 2),
        'base_accuracy':  round(base_acc, 2),
        'delta':          round(rag_acc - base_acc, 2),
        'mean_top_score': round(float(scores.mean()), 4) if len(scores) > 0 else None,
        'rag_helped':     int((results['is_correct_rag'] & ~results['is_correct_base']).sum()),
        'rag_hurt':       int((~results['is_correct_rag'] & results['is_correct_base']).sum()),
        'timestamp':      str(date.today()),
    }
    (run_dir / 'config.json').write_text(json.dumps(config, indent=2))

    rag_f.unlink()
    base_f.unlink()
    print(f'  ✓ RAG={rag_acc:.1f}%  Base={base_acc:.1f}%  Delta={rag_acc - base_acc:+.1f}pp')

print('\n✓ Repair complete.')

> Merging qwen3-embedding-0.6b__qwen3-4b-thinking-2507...
  ✓ RAG=30.0%  Base=32.0%  Delta=-2.0pp

✓ Repair complete.


In [4]:
for run_dir in sorted((REPO_ROOT / 'models' / 'evaluation').iterdir()):
    if not run_dir.is_dir():
        continue
    results_path = run_dir / 'results.parquet'
    config_path  = run_dir / 'config.json'

    if not results_path.exists():
        print(f'✗ {run_dir.name} — results.parquet missing')
        continue

    df  = pd.read_parquet(results_path)
    cfg = json.loads(config_path.read_text()) if config_path.exists() else {}

    has_rag  = 'is_correct_rag'  in df.columns
    has_base = 'is_correct_base' in df.columns

    status = '✓' if (has_rag and has_base) else '✗'
    cols   = list(df.columns)
    print(f'{status} {run_dir.name}')
    print(f'    rows={len(df)}  cols={cols}')
    if has_rag and has_base:
        print(f'    RAG={df["is_correct_rag"].mean()*100:.1f}%  '
              f'Base={df["is_correct_base"].mean()*100:.1f}%  '
              f'Delta={cfg.get("delta", "n/a")}pp')
    else:
        missing = [c for c in ['is_correct_rag', 'is_correct_base'] if c not in df.columns]
        print(f'    Missing columns: {missing}')

✗ octen-embedding-0.6b__qwen3-4b
    rows=99  cols=['question', 'correct', 'pred', 'is_correct', 'top_score', 'latency_s']
    Missing columns: ['is_correct_rag', 'is_correct_base']
✓ qwen3-embedding-0.6b__qwen3-4b-thinking-2507
    rows=100  cols=['question', 'correct', 'pred_rag', 'is_correct_rag', 'top_score', 'latency_s_rag', 'is_correct_base', 'pred_base', 'latency_s_base']
    RAG=30.0%  Base=32.0%  Delta=-2.0pp
✗ qwen3-embedding_0.6__qwen3-4b
    rows=50  cols=['question', 'correct', 'specialty', 'n_chunks', 'top_score', 'top_confidence', 'rag_pred', 'rag_correct', 'rag_latency', 'base_pred', 'base_correct', 'base_latency']
    Missing columns: ['is_correct_rag', 'is_correct_base']
